In [1]:
import asyncio
import pandas as pd
import os
from datetime import datetime, date
from ib_insync import *

from Class_xlWings import *
xlw  = xlWings()

In [2]:
def file_maintenance(xlApp, save_date):
    core_path = r"C:\Users\micha\Market_Data_Pulls"
    core_filename = "ETF_Options_"
    filename_ext = ".xlsx"

    open_filename = core_filename + "Template" + filename_ext
    open_address = os.path.join(core_path, open_filename)

    wb = xlApp.books.open(open_address)

    save_filename = core_filename + save_date.strftime('%Y%m%d') + "1" + filename_ext
    save_address = os.path.join(core_path, save_filename)

    wb.save(save_address)

    rtn_filename = os.path.basename(save_address)

    return wb, rtn_filename

In [3]:
async def get_option_snapshots(symbol="IBIT"):

    stock = Stock(symbol, "SMART", "USD")
    await ib.qualifyContractsAsync(stock)

    # --- underlying price ---
    ut = ib.reqMktData(stock, "", snapshot=True)
    await asyncio.sleep(1)
    underlying_price = ut.marketPrice()

    # --- option chain ---
    chains = await ib.reqSecDefOptParamsAsync(
        stock.symbol, "", stock.secType, stock.conId
    )
    chain = next(c for c in chains if c.exchange == "SMART")

    expirations = sorted(chain.expirations)#[:2]
    #print(expirations, "\n")
    strikes = [
        s for s in sorted(chain.strikes)
        if abs(s - underlying_price) <= 5
    ]

    contracts = [
        Option(
            symbol,
            exp,
            strike,
            right,
            chain.exchange,
            tradingClass=chain.tradingClass,
            currency="USD"
        )
        for exp in expirations
        for strike in strikes
        for right in ("C", "P")
    ]

    # 🔑 filters out invalid contracts
    contracts = await ib.qualifyContractsAsync(*contracts)

    tickers = [ib.reqMktData(c, "", snapshot=True) for c in contracts]
    await asyncio.sleep(2)

    rows = []
    for t in tickers:
        rows.append({
            "instrument_name": symbol,
            "option_type": t.contract.right,
            "strike": t.contract.strike,
            "expiration": t.contract.lastTradeDateOrContractMonth,
            "underlying_price": underlying_price,
            "mark_price": t.marketPrice(),
            "bid": t.bid,
            "ask": t.ask,
            "iv": t.modelGreeks.impliedVol if t.modelGreeks else None,
            "delta": t.modelGreeks.delta if t.modelGreeks else None,
            "gamma": t.modelGreeks.gamma if t.modelGreeks else None,
            "vega": t.modelGreeks.vega if t.modelGreeks else None,
            "theta": t.modelGreeks.theta if t.modelGreeks else None,
        })


    return pd.DataFrame(rows)

In [4]:
current_date_nyc = date.today()
current_time_nyc = str(datetime.now().time())

channel = int(current_time_nyc[0:2] + current_time_nyc[3:5] + current_time_nyc[6:8]) 

ib = IB()
await ib.connectAsync("127.0.0.1", 7496, clientId=channel)

xlApp = xw.App(visible=False)

wb, save_filename = file_maintenance(xlApp, current_date_nyc)

etf_list = ['IBIT', 'FBTC', 'GBTC', 'BTC', 'BITB', 'ARKB']
for etf in etf_list:
    df = await get_option_snapshots(etf)
    xlw.printDFToXL(save_filename, etf, 'a1', df)        

wb.save()
wb.close()

ib.disconnect()

Error 200, reqId 88: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260130', strike=49.5, right='C', exchange='SMART', currency='USD', tradingClass='IBIT')
Error 200, reqId 89: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260130', strike=49.5, right='P', exchange='SMART', currency='USD', tradingClass='IBIT')
Error 200, reqId 92: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260130', strike=50.5, right='C', exchange='SMART', currency='USD', tradingClass='IBIT')
Error 200, reqId 93: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260130', strike=50.5, right='P', exchange='SMART', currency='USD', tradingClass='IBIT')
Error 200, reqId 96: No security definition has been found for the request, contract: Op